# Analyse Helios Simulations
This notebook walks through the process of analysing helios simulations which follows 3 main steps:

    (1) Data Preparation
        This stage will setup the project directory, setup expected schemas for dataframes (both dask and pandas), and ultimately read in the helios data and prepare the required per ray information into a .parquet output.
        It will also setup the reference dataset for voxels for each voxel_size in the project (i.e. unique voxel_ids etc.).
    
    (2) Voxel-Ray Intersection
        With valid rays saved per leg of the scan, in the previous step, the goal now is to check ray intersections in all voxels. This will record important information, such as the entry/exit/hit coordinates of the ray which will later be used to gather metrics.
        The main reason these metrics are not gathered yet, is that this stage will remain separate per leg. That way, the metrics can be computed from different combinations of helios legs without re-computing voxel-ray intersections.

    (3) Compute Metrics
        Taking a given set of legs and voxel_sizes, the voxel_ray intersection files will be used to calculate metrics for each voxel, in this case resulting in all outputs from each investigated method.

# Step 1 - Setup Project
Set project paths here

In [1]:
import os

# Set up the project directory
project_dir = '/home/capheus/projects/1001_etri_erecto_diamond'
helios_dir = os.path.join(project_dir, 'helios')
references_dir = os.path.join(project_dir, 'references')
results_dir = os.path.join(project_dir, 'results')
valid_rays_dir = os.path.join(project_dir, 'valid_rays')

if not os.path.exists(helios_dir) or not os.path.exists(references_dir):
    raise FileNotFoundError("The specified directories do not exist. Please check the paths.")

if not os.path.exists(valid_rays_dir):
    os.makedirs(valid_rays_dir, exist_ok=True)

if not os.path.exists(results_dir):
    os.makedirs(results_dir, exist_ok=True)

use_class = True
leaf_object_ids = [1]
wood_object_ids = [0]


## Step 1 - Data Preparation
This step focuses on converting helios simulation outputs, saving only valid rays into a more efficient .parquet file format.

It expects the following input and will add a new folder (valid_rays) to store all resulting .parquet files.

INPUT:
    project_dir/
    ├── reference/
    │   ├── "{project}_voxel_size_0.2.csv"
    │   ├── "{project}_voxel_size_0.5.csv"
    │   ...
    │   └── "{project}_voxel_size_{v}.csv"
    ├── helios/
    │   ├── "leg000_points.xyz"
    │   ├── "leg000_pulse.txt"
    │   ├── "leg000_fullwave.txt"
    │   ├── "leg001_points.xyz"
    │   ├── "leg001_pulse.txt"
    │   ├── "leg001_fullwave.txt"
    │   ├── ...
    │   ├── "leg{l}_points.xyz"
    │   ├── "leg{l}_pulse.txt"
    │   └── "leg{l}_fullwave.txt"

OUTPUT:
    └── valid_rays/
        ├── "leg_000_valid_rays.parquet"
        ├── "leg_001_valid_rays.parquet"
        ...
        └── "leg_{l}_valid_rays.parquet"

In [ ]:
from utils import prepare_helios_data

# Run the data preparation script
prepare_helios_data(
    input_dir=helios_dir, 
    output_dir=valid_rays_dir, 
    references_dir=references_dir, 
    leaf_object_ids=leaf_object_ids, 
    wood_object_ids=wood_object_ids, 
    use_class=use_class,
    debug=True
)

### Step 1.5 -  Compute Normals and Weights for Leaf Points

In [ ]:
from utils import add_normals_weights_to_valid_rays

# Calculate normals and weights by loading valid rays
add_normals_weights_to_valid_rays(
    valid_rays_dir, 
    debug=True,
    knn=6
)

## Step 2 - Voxel Ray Intersections
This code uses the valid rays from before, alongside the reference datasets in order to create a supporting parquet in the valid rays folder using the voxel_size_{voxel_size}_leg_{leg}_intersections.parquet format.

In [2]:
from utils import voxel_ray_intersections

# Run intersections
voxel_ray_intersections(
    valid_rays_dir=valid_rays_dir, 
    references_dir=references_dir,
    debug=False
)

[voxel_ray_intersections] Initialising Dask client...
No SLURM_CPUS_PER_TASK detected, using system CPU count: 11 physical cores with 2 threads per worker.
Using OS temporary directory: /tmp
[voxel_ray_intersections] Starting Dask with memory_limit=4286MB
Found 1 voxel reference files.
Compiled voxel references with 5 entries.
Found 12 valid rays files.
[voxel_ray_intersections] Loading valid rays file for leg 4: /home/capheus/projects/1001_etri_erecto_diamond/valid_rays/leg_4_valid_rays.parquet
[voxel_ray_intersections] Leg 4 partitions: 6
[voxel_ray_intersections] Mapped partitions for leg 4
[voxel_ray_intersections] Loading valid rays file for leg 3: /home/capheus/projects/1001_etri_erecto_diamond/valid_rays/leg_3_valid_rays.parquet
[voxel_ray_intersections] Leg 3 partitions: 6
[voxel_ray_intersections] Mapped partitions for leg 3
[voxel_ray_intersections] Loading valid rays file for leg 2: /home/capheus/projects/1001_etri_erecto_diamond/valid_rays/leg_2_valid_rays.parquet
[voxel_ra

2025-12-03 06:25:45,208 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:45971' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('repartitiontofewer-b071abae48cd377dc853314b4fe50579', 0), ('repartitiontofewer-aaad5e872fdb37c0da709b6c4f62ed9e', 0), ('repartitiontofewer-1278736ccc448ae911db63390361ee6e', 0)} (stimulus_id='handle-worker-cleanup-1764707145.208451')
2025-12-03 06:25:45,211 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:36801' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('repartitiontofewer-4228d5e03804ccb6cb7cca4c11c16abb', 0)} (stimulus_id='handle-worker-cleanup-1764707145.2110844')
2025-12-03 06:25:45,213 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:35493' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('repartitiontofewer-f86bbb512426b73d0ae9c66ed0582604', 0)} (stimulus_id='han

Deleted Dask worker scratch space at /tmp/dask-scratch-space
[voxel_ray_intersections] Dask client closed.


## Step 3 - Compute Metrics
Using the leg_{leg_id}_voxel_size_{voxel_size}_intersections.parquet files (which feature a standardised structure of columns from various inputs), compute the desired metrics and save outputs.

In [ ]:
import os
import glob
import utils
import pandas as pd
from utils import calculate_lambda_1, get_voxel_metrics

# Select the desired legs and voxel_sizes to include in the analysis
# Use the shortcut string 'all' to include all 
legs = 'all' # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
voxel_sizes = [2.0] #'all' # [0.2, 0.5, 1.0, 2.0]

leg_string = None if not legs == 'all' else legs

# Set the average leaf area
average_leaf_area = 0.003582  # in m^2, adjust as needed

# Get the list of all voxel sizes
intersection_files = []
if legs == 'all' and voxel_sizes == 'all':
    intersection_files = glob.glob(os.path.join(valid_rays_dir, '*_intersections.parquet'))
elif legs == 'all' and isinstance(voxel_sizes, list):
    for voxel_size in voxel_sizes:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet'))
elif isinstance(legs, list) and voxel_sizes == 'all':
    for leg in legs:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_*_intersections.parquet'))
else:
    for leg in legs:
        for voxel_size in voxel_sizes:
            intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_voxel_{voxel_size}_intersections.parquet'))

# Check if any intersection files were found
if intersection_files == []:
    print("No intersection files found. Please check the input parameters.")

# Split intersection files into separate lists for each voxel_size
voxel_size_files = {}
for file in intersection_files:
    # Extract the voxel size from the filename
    parts = file.split('_')
    voxel_size = float(parts[parts.index('voxel') + 1])
    
    # Add the file to the corresponding voxel size list
    if voxel_size not in voxel_size_files:
        voxel_size_files[voxel_size] = []
    voxel_size_files[voxel_size].append(file)

# Extract voxel information for each voxel size
for voxel_size, files in voxel_size_files.items():
    # Create a list of all legs in files
    legs = []
    for file in files:
        leg = os.path.basename(file)
        parts = leg.split('_')
        leg = int(parts[parts.index('leg') + 1])
        legs.append(leg)

    # Calculate the lambda_1 for average leaf area
    lambda_1 = calculate_lambda_1(voxel_size=voxel_size, average_leaf_area=average_leaf_area)
    print(f"Voxel size: {voxel_size}, Lambda_1: {lambda_1}")

    # Calculate per voxel information from all files
    voxel_metrics_df = get_voxel_metrics(
        intersections_files=files, 
        lambda_1=lambda_1,
        is_multireturn=False
    )

    # Retrieve the reference file
    reference_file = glob.glob(os.path.join(references_dir, f'*{voxel_size}*'))[0]
    df_ref = pd.read_csv(reference_file)

    # CI_leaf_Corr, CI_lw_Corr
    # Ensure only numeric columns are included in the mean operation
    if 'voxel_id' in df_ref.columns:
        df_ref = df_ref.groupby('voxel_id').mean(numeric_only=True).reset_index()
        df_ref = df_ref.add_suffix('_ref')
    elif 'voxel_cx' in df_ref.columns:
        df_ref = df_ref.groupby(['voxel_cx', 'voxel_cy', 'voxel_cz']).mean(numeric_only=True).reset_index()
        df_ref = df_ref.add_suffix('_ref')

    df_ref.rename(columns={
        'voxel_id_ref': 'voxel_id',
        'LAD_ref_ref': 'LAD_ref', 
        'PAD_ref_ref': 'PAD_ref'
        }, inplace=True)

    for c in ['voxel_cx' ,'voxel_cy', 'voxel_cz']:
        if c + '_ref' in df_ref.columns:
            df_ref.rename(columns={c + '_ref': c}, inplace=True)

    # Merge to maintain voxel_id matching
    if 'voxel_id' in df_ref.columns:
        voxel_metrics_df = voxel_metrics_df.merge(df_ref, on='voxel_id', how='left')

        if 'voxel_cx' in voxel_metrics_df.columns:
            voxel_metrics_df.drop(columns=['voxel_cx'], inplace=True)
        if 'voxel_cy' in voxel_metrics_df.columns:
            voxel_metrics_df.drop(columns=['voxel_cy'], inplace=True)
        if 'voxel_cz' in voxel_metrics_df.columns:
            voxel_metrics_df.drop(columns=['voxel_cz'], inplace=True)
            
    elif 'voxel_cx' in df_ref.columns:
        voxel_metrics_df = voxel_metrics_df.merge(df_ref, on=['voxel_cx', 'voxel_cy', 'voxel_cz'], how='left')
    

    ### Add LAD calculations here if desired
    """Example, LAD_BL_TLS

    # Retrieve required variables
    I_leaf = voxel_metrics_df['I_leaf'].values
    mean_path_length = voxel_metrics_df['mean_path_length'].values  
    G_leaf = voxel_metrics_df['G_leaf'].values
    CI_leaf_ref = voxel_metrics_df['CI_leaf_corr_ref'].values

    LAD_BL_TLS = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length)
    LAD_BL_TLS_G = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf)
    LAD_BL_TLS_CI_ref = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf, CI=CI_leaf_ref)
    """

    # Save outputs to csv
    project_name = os.path.basename(os.path.normpath(project_dir))
    if leg_string is None:
        legs.sort()
        leg_string = "_".join(map(str, legs))
    output_file = os.path.join(results_dir, f"{project_name}_leg_{leg_string}_voxel_size_{voxel_size}.csv")
    if os.path.exists(output_file):
        os.remove(output_file)
    voxel_metrics_df.to_csv(output_file)

# Extra Tools

In [ ]:
import glob
from utils import convert_parquet_to_csv

voxel_sizes = [2.0] #[0.2, 0.5, 1.0, 2.0]
for voxel_size in voxel_sizes:
    input_files = glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet'))
    if input_files:
        for input_file in input_files:
            output_file = input_file.replace('.parquet', '.csv')
            convert_parquet_to_csv(input_file, output_file)
            print(f"Converted {input_file} to {output_file}")